# 🚗 YOLO ile Plaka Tespiti (Plate Detection) Model Eğitimi - Google Colab

Bu notebook, **Yolo_Plate_Detection** pipeline'ından elde ettiğiniz veri seti ile Google Colab'in ücretsiz GPU desteğini (T4 GPU) kullanarak YOLOv8 / YOLO11 plaka tespit modeli eğitmenizi sağlar.

> **İpucu:** Colab menüsünden `Düzenle` -> `Not Defteri Ayarları` -> `Donanım Hızlandırıcı: T4 GPU` seçili olduğundan emin olun.

## 1. GPU Kontrolü
GPU'nun etkinleştirildiğini doğrulamak için aşağıdaki hücreyi çalıştırın:

In [ ]:
!nvidia-smi

## 2. Gerekli Kütüphanelerin Kurulumu
Ultralytics kütüphanesini en güncel sürümüyle kuralım:

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

## 3. Veri Setini Yükleme
Uygulamamızın 5. adımında oluşturduğu `dataset` klasörünü zip olarak sıkıştırıp Colab'e yükleyin veya Google Drive üzerinden bağlayın.

In [ ]:
# Seçenek A: Google Drive Bağlama (Önerilen)
from google.colab import drive
drive.mount('/content/drive')

# Drive'daki dataset.zip dosyasını /content dizinine çıkarın:
# !unzip -q /content/drive/MyDrive/dataset.zip -d /content/dataset

# Seçenek B: Doğrudan Colab'e zip yüklediyseniz:
# !unzip -q /content/dataset.zip -d /content/dataset

## 4. data.yaml Yapılandırma Dosyası Oluşturma
YOLO'nun veri setini okuyabilmesi için `data.yaml` dosyasını otomatik oluşturalım:

In [ ]:
import yaml

dataset_config = {
    'path': '/content/dataset',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'plate'
    }
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("data.yaml başarıyla oluşturuldu:")
!cat /content/data.yaml

## 5. Modeli Eğitme (Training)
YOLOv8n (nano) veya YOLOv8s (small) ağırlıklarını transfer learning ile eğitebilirsiniz:

In [ ]:
from ultralytics import YOLO

# YOLOv8 nano modelini başlangıç ağırlıklarıyla yükleyin (veya yolov8s.pt, yolo11n.pt)
model = YOLO('yolov8n.pt')

# Eğitimi başlatın
results = model.train(
    data='/content/data.yaml',
    epochs=50,          # İhtiyacınıza göre artırabilirsiniz (örn: 100)
    imgsz=640,          # Görsel çözünürlüğü
    batch=16,           # GPU RAM durumuna göre 16 veya 32
    device=0,           # GPU cihazı (0)
    workers=2,
    save=True,
    project='plate_detector',
    name='yolov8_plate_run'
)

## 6. Eğitim Sonuçlarını Değerlendirme (Validation & Metrics)
Eğitim tamamlandıktan sonra mAP50, mAP50-95, precision ve recall metriklerini inceleyin:

In [ ]:
from IPython.display import Image, display

# Kayıp (loss) ve doğruluk grafiklerini görüntüleme
display(Image('/content/plate_detector/yolov8_plate_run/results.png'))

# Karmaşıklık matrisi (Confusion Matrix)
display(Image('/content/plate_detector/yolov8_plate_run/confusion_matrix.png'))

## 7. Test Görselleri Üzerinde Tahmin (Inference)
En iyi ağırlığı (`best.pt`) kullanarak test veri setinde tahmin yapalım:

In [ ]:
best_model = YOLO('/content/plate_detector/yolov8_plate_run/weights/best.pt')

# Test klasöründen tahmin al
pred_results = best_model.predict(
    source='/content/dataset/images/test',
    conf=0.25,
    save=True,
    project='plate_detector',
    name='test_predictions'
)

print("Tahminler kaydedildi: /content/plate_detector/test_predictions")

## 8. Eğitilmiş Modeli İndirme ve Dışa Aktarma
Eğitilen en iyi modeli (`best.pt`) bilgisayarınıza indirin veya ONNX formatına dönüştürün:

In [ ]:
from google.colab import files

# Modeli ONNX formatına aktar (Opsiyonel: C++, TensorRT veya web entegrasyonu için)
best_model.export(format='onnx')

# best.pt dosyasını yerel bilgisayarınıza indirin:
files.download('/content/plate_detector/yolov8_plate_run/weights/best.pt')